In [108]:
import aiohttp
import asyncio
from bs4 import BeautifulSoup

from datetime import datetime as dt
import json

# Extract all article links

In [3]:
start_month = 2
start_year = 2012

url = 'https://orthosphere.wordpress.com/'

In [6]:
res = requests.get(url + str(start_year) + '/' + str(start_month) + '/')

In [26]:
session = aiohttp.ClientSession()

In [33]:
async def get_article_links(url):
    
    res = await session.get(url)
    bs = BeautifulSoup(await res.text())
    article_links = [ar.find('h1', class_='entry-title').find('a')['href'] for ar in bs.find_all('article')]
    
    return article_links

In [51]:
tasks = []
dates = []

y = start_year
m = start_month

now = dt.now()

while y != now.year or m != now.month+1:
    

    
    date = str(y) + '/' + str(m)
    
    print(f'Getting articles for {date}...')
    
    dates.append(date)
    tasks.append(get_article_links(url + date + '/'))
    
    
    if m == 12:
        m = 1
        y += 1
    else:
        m += 1

Getting articles for 2012/2...
Getting articles for 2012/3...
Getting articles for 2012/4...
Getting articles for 2012/5...
Getting articles for 2012/6...
Getting articles for 2012/7...
Getting articles for 2012/8...
Getting articles for 2012/9...
Getting articles for 2012/10...
Getting articles for 2012/11...
Getting articles for 2012/12...
Getting articles for 2013/1...
Getting articles for 2013/2...
Getting articles for 2013/3...
Getting articles for 2013/4...
Getting articles for 2013/5...
Getting articles for 2013/6...
Getting articles for 2013/7...
Getting articles for 2013/8...
Getting articles for 2013/9...
Getting articles for 2013/10...
Getting articles for 2013/11...
Getting articles for 2013/12...
Getting articles for 2014/1...
Getting articles for 2014/2...
Getting articles for 2014/3...
Getting articles for 2014/4...
Getting articles for 2014/5...
Getting articles for 2014/6...
Getting articles for 2014/7...
Getting articles for 2014/8...
Getting articles for 2014/9...
Ge

/tmp/ipykernel_901172/3852388888.py:1: RuntimeWarning: coroutine 'get_article_links' was never awaited
  tasks = []


In [52]:
results = await asyncio.gather(*tasks)

In [98]:
urls = results

# Scrape content

In [56]:
async def get_content(url):
    
    res = await session.get(url)
    bs = BeautifulSoup(await res.text())
    
    out = bs.find('div', class_='content-area', id='primary')
    
    return out

In [ ]:
async def get_article(content):
    
    article = content.find('article')
    
    title = article.find('h1', class_='entry-title').text
    date = content.find('time', class_='entry-date')['datetime']
    author = content.find('span', class_='author vcard').a.text
    text = content.find('div', class_='entry-content').text
    html = content.find('div', class_='entry-content')
       
    return {
        'title': title,
        'date': date,
        'author': author,
        'html': str(html),
        'text': text
    }

In [103]:
async def get_comments(content):
    
    comment_section = content.find_all('ol', class_='commentlist')
    
    comments = comment_section[0].find_all('article', class_='comment')
    
    out = []
    
    for comment in comments:
        # print("NEW:", comment)
        author = comment.find_all('div', class_='comment-author vcard')
        author = author[0].find('cite').text if author else 'Unknown'
        text = comment.find_all('div', class_='comment-content')[0].text
        time = comment.find('time')['datetime']
        
        out.append({
            'author': author,
            'text': text,
            'time': time
        })
        
    return out

In [106]:
async def scrape_all(dates, urls):
    
    full_out = {}
    
    for dt, url_list in zip(dates, urls):
        print(f'Scraping {dt}...')
        
        dt_out = []
        
        for url in url_list:
            print(f'Processing {url}...')
            
            try:
                content = await get_content(url)
                article = await get_article(content)
                comments = await get_comments(content)
            
                dt_out.append({
                    'date': dt,
                    'url': url,
                    'article': article,
                    'comments': comments
                })
            except Exception as e:
                print(f'\tError processing {url}: {e}')
                dt_out.append({
                    'date': dt,
                    'url': url,
                    'error': str(e)
                })
            
        full_out[dt] = dt_out
        
    return full_out

In [107]:
all_contents = await scrape_all(dates, urls)

Scraping 2012/2...
Processing https://orthosphere.wordpress.com/2012/02/29/another-unprincipled-exception-bites-the-dust/...
Processing https://orthosphere.wordpress.com/2012/02/28/arn-the-knight-templar/...
Processing https://orthosphere.wordpress.com/2012/02/28/how-many-saved-how-many-damned/...
Processing https://orthosphere.wordpress.com/2012/02/28/credo-before-all-worlds/...
Processing https://orthosphere.wordpress.com/2012/02/27/too-good-to-pass-up/...
Processing https://orthosphere.wordpress.com/2012/02/27/celebrated-pedophile-exposed-as-enemy-of-the-revolution/...
Processing https://orthosphere.wordpress.com/2012/02/26/the-audacity-of-natural-law/...
Processing https://orthosphere.wordpress.com/2012/02/26/repost-two-hymns/...
Processing https://orthosphere.wordpress.com/2012/02/26/the-monumental-hubris-of-the-modern-heretic/...
Processing https://orthosphere.wordpress.com/2012/02/25/the-continuing-decline-of-the-church-of-england/...
Scraping 2012/3...
Processing https://orthos

In [1]:
# str(all_contents['2012/10'][0]['article']['html'])

In [ ]:
# for k in all_contents.keys():
#     for k2 in range(len(all_contents[k])):
#         if 'article' in all_contents[k][k2]:
#             all_contents[k][k2]['article']['html'] = str(all_contents[k][k2]['article']['html'])

In [133]:
json.dump(all_contents, open('orthosphere_data_raw.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=4)

In [134]:
# TODO: fix errors in some articles